In [1]:
%pip install "torch>=2.0.0" "transformers>=5.0.0" scikit-learn tqdm plotly pandas ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import json
import time

import torch
from transformers import pipeline
from tqdm.notebook import tqdm
import plotly.express as px
import pandas as pd
from sklearn.decomposition import PCA

In [3]:
# Load list of subject terms for processing

with open('input/subject_terms.txt', 'r') as f:
    subject_terms = [line.strip() for line in f.readlines() if line.strip()]

subject_terms[:5]

['20-minute makeover', '25 year club', '3 rs', '3-1-1', '311']

In [4]:
# Load the categories and their descriptions
#
# Categories are stored as json objects:
# [{ "name": "...", "description": "..." }, ...]

with open('input/categories.json', 'r') as f:
    categories_json = json.load(f)

categories_short = [cat['name'] for cat in categories_json]
# Target input format: "<name>: <description>"
categories_verbose = [cat['name'] + ': ' + cat['description'] for cat in categories_json]

categories_verbose[:5]

['Accessibility: This is about accessibility for people with disabilities (the disabled), such as wheel-trans, wheelchairs, mobility aids, accessible parking permits, AODA, barrier-free design, accessible taxis, accessible parking, and mobility devices. Inputs that contain accessible or disabled as descriptors belong here, NOT in Governance & City Hall.',
 'Animals & Wildlife: Concerns about wild animals in urban or suburban areas, including animal sightings, injured or sick wild animals, and wildlife removal or deterrence. Covers all wild animals including raccoons, coyotes, skunks, snakes, squirrels, bats, deer, foxes, chickens, and wild birds (geese, gulls, pigeons). Also covers integrated pest management programs for rodents and invasive species. Excludes domestic pets (cats and dogs). Excludes natural habitat preservation.',
 'Arts, Culture & Events: This is about arts, culture, events, and tourism, such as Canadian National Exhibition (CNE), festivals, museums, public art, theatr

In [5]:
# Detect processing device and estimate batch size
# Values for model_batch_size are safe defaults, and can be increased to utilize more memory
import os

if torch.cuda.is_available():
    device = "cuda" # GPU
    # Estimate batch size based on GPU VRAM
    free_bytes, _ = torch.cuda.mem_get_info()
    free_gb = free_bytes / (1024 ** 3)
    model_batch_size = min(max(int(free_gb * 1024 / 50), 8), 128) # min: 8, max: 128
elif torch.backends.mps.is_available():
    device = "mps" # Apple silicon
    # Estimate batch size based on system memory
    total_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024 ** 3)
    if total_gb >= 32:
        model_batch_size = 32
    elif total_gb >= 16:
        model_batch_size = 16
    else:
        model_batch_size = 8
else:
    device = "cpu"
    model_batch_size = 8

print(f"Using device {device} with batch_size {model_batch_size}")

Using device mps with batch_size 16


In [6]:
# Load model
clf = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0",
    device=device,
    batch_size=model_batch_size
)

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

In [7]:
# Answering a question by garo - this is a test of the model
# result = clf("assault", ['crime', 'safety', 'climate'], multi_label=True)
# result

In [8]:
#this is another test of the model
# seq = "hello world!"
# candidate_labels = ["greeting", "food"]
# result = clf(seq, candidate_labels, multi_label=True)
# result

In [9]:
#test case
# TODO: seq not used, remove var or entire block?
# TODO: The action inference is done slightly differently (eg. hypothesis), do we want consistency?
# seq = "3-1-1"
# candidate_labels = categories_verbose
# result = clf(subject_terms[:100], candidate_labels, multi_label=True)
# result

In [10]:
#Commented out to make the notebook easier to read
# dict(zip(result[0]['labels'], result[0]['scores']))

In [ ]:
# Compute similarities
# This is where the model uses the descriptions to assign each tag to a category.
# The hypothesis template and the context have been added to halp map the tags to categories.
# This is the most time consuming part, taking over 2 hours on TH laptop.
filename = 'similarities.csv'
hypothesis_template = "In Toronto municipal government, this topic relates to {}."
input_terms=subject_terms[:] # Adjust this to limit the input set

with open(filename, 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=['subject_term'] + categories_short)
    writer.writeheader()

    batch_index = 0
    batch_size = 200
    progress = tqdm(total=len(input_terms))
    while batch_index * batch_size < len(input_terms):
        start = time.perf_counter()
        curr_term_batch = input_terms[batch_index * batch_size:(batch_index + 1) * batch_size]
        contextualized_batch = [f"Toronto City Council agenda topic: {t}" for t in curr_term_batch]

        progress.write(f"Starting batch {batch_index}/{(len(input_terms) + batch_size - 1) // batch_size}")

        results = clf(contextualized_batch, categories_verbose, multi_label=True, hypothesis_template=hypothesis_template)
        output = []
        for result, original_term in zip(results, curr_term_batch):
            score_map = dict(zip(result['labels'], result['scores']))
            row = {short: score_map[verbose] 
                   for short, verbose in zip(categories_short, categories_verbose)}
            row['subject_term'] = original_term  # ← keep the ORIGINAL term, not the contextualized one
            output.append(row)

        writer.writerows(output)
        elapsed = time.perf_counter() - start
        progress.write(f"Completed batch {batch_index} in {elapsed:.2f} seconds")
        progress.update(len(curr_term_batch))
        batch_index += 1

    progress.close()

  0%|          | 0/3717 [00:00<?, ?it/s]

Starting batch 1/19
Completed batch 0 in 267.42 seconds
Starting batch 2/19
Completed batch 1 in 268.23 seconds
Starting batch 3/19
Completed batch 2 in 263.45 seconds
Starting batch 4/19


In [ ]:
def reduce_dimensionality(df: pd.DataFrame, num_dimensions: int) -> pd.DataFrame:
    pca = PCA(n_components=num_dimensions)
    pca.fit(df)
    reduced_df = pd.DataFrame(pca.transform(df))
    return reduced_df

In [ ]:
df = pd.read_csv('similarities.csv')
df['max_val'] = df.drop(columns=['subject_term']).max(axis=1)
df['max_cat'] = df.drop(columns=['subject_term']).idxmax(axis=1)
df

In [ ]:
reduced_df = pd.DataFrame(df)
reduced_df.drop(columns=['subject_term', 'max_val', 'max_cat'], inplace=True)
reduced_df = reduce_dimensionality(reduced_df, 3)
reduced_df.insert(0, 'subject_term', pd.Series(df['subject_term']))
reduced_df

In [ ]:
#fig = px.scatter_3d(reduced_df, x=0, y=1, z=2, hover_name='subject_term')
#fig.show()

In [ ]:
#fig = px.scatter_matrix(df, dimensions=categories_short[:5], width=1600, height=800, hover_name='subject_term')
#fig.show()

In [ ]:
print(df.shape)  # how many rows are going into fit_transform?

In [ ]:
# Clear model from memory, so it can be reclaimed
# Useful for memory-constrained devices, not strictly necessary

del clf
import gc
gc.collect()

import torch
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()

In [ ]:
# ── Post-processing: rule-based corrections ──────────────────────────────────

# Load rules from file
with open('input/post-processing-rules.json', 'r') as f:
    rules = json.load(f)

for i, rule in enumerate(rules, 1):
    pattern = rule['pattern']
    category = rule['category']

    mask = (
        df['subject_term'].str.contains(pattern, case=False, regex=True) &
        (df['max_val'] < 1.0) &
        (df['max_cat'] != category)
    )
    df.loc[mask, 'max_cat'] = category
    df.loc[mask, 'max_val'] = 1.0
    print(f"Rule {i} | Category: {category} Pattern: {pattern} | {mask.sum()} terms corrected")

In [ ]:
# Display top candidate subject terms for each category
threshold = 0.005
top_terms = df[df['max_val'] >= threshold]
#top_terms

In [ ]:
top_terms.to_csv("top_terms.csv")

In [ ]:
df.to_csv("all_terms.csv")

In [ ]:
# Generate histograms of similarities per category
import plotly.graph_objects as go
import numpy as np

# fig = px.histogram(
#     df, 
#     x='housing',
#     width=500,
#     height=500,
#     nbins=20
# )
fig = go.Figure()
for cat in categories_short:
    fig.add_trace(go.Histogram(
        x=df[cat],
        xbins=dict(
            start=0.0,
            end=1.0,
            size=0.05
        ),
        name=cat.capitalize()
    ))
fig.update_layout(barmode='overlay', width=500)
fig.update_traces(opacity=0.75)
fig.show()

In [ ]:
fig = px.histogram(
    df,
    x='max_val',
    color='max_cat',
    marginal='rug',
    hover_name='max_cat',
    width=1000,
    height=750,
    nbins=10,
    title='Most similar category of agenda items\' subject terms',
)
fig.show()

In [ ]:
#for cat in categories_short:
#    fig = px.histogram(
#        df,
#        x=cat,
#        marginal='rug',
#        nbins=20,
#        width=500,
#        height=500,
#        # color='max_cat'
#    )
#    fig.show()

In [ ]:
# FIX: Use 'max_val' instead of a specific category name
# This shows you the distribution of your model's "Best Guesses"
counts, bins = np.histogram(df['max_val'], bins=[i / 100 for i in range(0, 101, 5)])
bins = .5 * (bins[:-1] + bins[1:])
fig = px.bar(x=bins, y=counts, title="Distribution of Top Confidence Scores")
fig.show()

In [ ]:
bins, counts

In [ ]:
histogram_table = []
for cat in categories_short:
    counts, bins = np.histogram(df[cat], bins=[i / 100 for i in range(0, 101, 5)])
    histogram_table.append(counts.tolist())
    # bins = .5 * (bins[:-1] + bins[1:])
    # fig = px.bar(x=bins, y=counts)
    # fig.show()
histogram_table = pd.DataFrame(
    histogram_table,
    columns=[
        "0.00 - 0.05",
        "0.05 - 0.10",
        "0.10 - 0.15",
        "0.15 - 0.20",
        "0.20 - 0.25",
        "0.25 - 0.30",
        "0.30 - 0.35",
        "0.35 - 0.40",
        "0.40 - 0.45",
        "0.45 - 0.50",
        "0.50 - 0.55",
        "0.55 - 0.60",
        "0.60 - 0.65",
        "0.65 - 0.70",
        "0.70 - 0.75",
        "0.75 - 0.80",
        "0.80 - 0.85",
        "0.85 - 0.90",
        "0.90 - 0.95",
        "0.95 - 1.00"
    ],
    index=categories_short
)
#histogram_table

In [ ]:
histogram_table.to_csv('histogram_table.csv')

In [ ]:
sorted(categories_short)

---

In [ ]:
from collections import Counter

# count_terms = Counter()
# with open('/kaggle/input/datasets/jeromegv/normalized-subject-terms-txt/normalized_subject_terms.txt', 'r', encoding='utf-8') as file:
#     for line in file:
#         line = line.strip()
#         count_terms[line] += 1

# Using the pre-loaded data
count_terms = Counter(subject_terms)

In [ ]:
sorted_terms = sorted(count_terms, reverse=True, key=lambda x: count_terms[x])

In [ ]:
for term in sorted_terms:
    print(term, count_terms[term])

In [ ]:
len(count_terms), df.shape

In [ ]:
df.iloc[4000:4005]['subject_term'], sorted(count_terms)[4000:4005]

In [ ]:
df['count'] = [count_terms[term] for term in sorted(count_terms)]
#df.iloc[4000:4005]

In [ ]:
count_terms['smell']

In [ ]:
df.to_csv('all_terms.csv')

In [ ]:
# load in a validation file and create a dict

#df = pd.read_csv('/Users/tynahope/Documents/civic dashboard tags/ValidationJune3_325.csv')
df  = pd.read_csv('/Users/tynahope/Documents/civic dashboard tags/GT-TAH-June8.csv')
validation_labels = dict(zip(df['subject_term'], df['expected_category']))

validation_terms = list(validation_labels.keys())
validation_expected = list(validation_labels.values())

print(f"Loaded {len(validation_labels)} terms across {len(set(validation_expected))} categories")

In [ ]:
# ── Validation: Zero-Shot Classification vs. Manual Labels ──────────────────
# Paste this cell AFTER the cells that define:
#   - validation_labels  (from validation_set.py)
#   - the top_terms.csv results (adjust path below if needed)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.metrics import classification_report, accuracy_score

# ── 1. Load classification results ──────────────────────────────────────────
results_path = "top_terms.csv"          # ← update path if needed
df_results = pd.read_csv(results_path)

# ── 2. Remap old category names in max_cat to current dict keys ─────────────
# (needed if the CSV was produced with a previous version of the categories)

df_results["predicted_category"] = df_results["max_cat"]

# ── 3. Build comparison dataframe from validation set ───────────────────────
df_val = pd.DataFrame(
    list(validation_labels.items()),
    columns=["subject_term", "expected_category"]
)

df_merged = df_val.merge(
    df_results[["subject_term", "predicted_category"]],
    on="subject_term",
    how="left"
)

missing = df_merged["predicted_category"].isna().sum()
if missing > 0:
    print(f"⚠️  {missing} validation term(s) not found in results — they will be excluded.")
    df_merged = df_merged.dropna(subset=["predicted_category"])

y_true = df_merged["expected_category"].tolist()
y_pred = df_merged["predicted_category"].tolist()

# ── 4. Overall accuracy ──────────────────────────────────────────────────────
acc = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {acc:.1%}  ({int(acc * len(y_true))}/{len(y_true)} correct)\n")

# ── 5. Per-category report ───────────────────────────────────────────────────
print(classification_report(y_true, y_pred, zero_division=0))

# ── 6. Confusion matrix (heatmap) ───────────────────────────────────────────
all_cats = sorted(set(y_true) | set(y_pred))
n = len(all_cats)
cat_index = {c: i for i, c in enumerate(all_cats)}

matrix = np.zeros((n, n), dtype=int)
for true, pred in zip(y_true, y_pred):
    matrix[cat_index[true]][cat_index[pred]] += 1

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(matrix, cmap="Blues")

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(all_cats, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(all_cats, fontsize=8)
ax.set_xlabel("Predicted category", fontsize=10)
ax.set_ylabel("True category (your labels)", fontsize=10)
ax.set_title(f"Confusion Matrix — Zero-Shot Classification\nOverall accuracy: {acc:.1%}", fontsize=12)

# Annotate cells that have values
for i in range(n):
    for j in range(n):
        val = matrix[i, j]
        if val > 0:
            colour = "white" if val > matrix.max() * 0.5 else "black"
            ax.text(j, i, str(val), ha="center", va="center",
                    fontsize=7, color=colour)

plt.colorbar(im, ax=ax, label="Count")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Confusion matrix saved to confusion_matrix.png")

# ── 7. Most common confusions (off-diagonal) ────────────────────────────────
print("\nTop 15 confusions (true → predicted):")
print(f"{'True category':<35} {'Predicted category':<35} {'Count':>5}")
print("-" * 77)
confusions = []
for i, true_cat in enumerate(all_cats):
    for j, pred_cat in enumerate(all_cats):
        if i != j and matrix[i, j] > 0:
            confusions.append((true_cat, pred_cat, matrix[i, j]))
confusions.sort(key=lambda x: -x[2])
for true_cat, pred_cat, count in confusions[:15]:
    print(f"{true_cat:<35} {pred_cat:<35} {count:>5}")

In [ ]:
df_merged.to_csv("validation_out.csv")


In [ ]:
score_lookup = df_results.set_index("subject_term")["max_val"]
df_merged["max_val"] = df_merged["subject_term"].map(score_lookup)
df_merged.to_csv("validation_out_withmaxval.csv")